In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install torchtext==0.16.0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import re
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import torch
import torch.nn as nn
from torch.nn import functional as F

# Dataset

In [ ]:
data_path = '/content/drive/MyDrive/Colab Notebooks/sentiment_analysis/data'

In [ ]:
import os
import pandas as pd
def load_data_from_path(file_path):
    df =  pd.read_csv(file_path)
    df.columns = ['Header1', 'company','label','text']
    df.dropna(inplace=True)
    df.drop_duplicates(inplace=True)
    df.drop(columns=['Header1', 'company'], inplace=True)
    return df
train_df = load_data_from_path(os.path.join(data_path, 'twitter_training.csv'))
test_df = load_data_from_path(os.path.join(data_path, 'twitter_validation.csv'))

In [ ]:
test_df.head()

,label,text
0,Neutral,BBC News - Amazon boss Jeff Bezos rejects clai...
1,Negative,@Microsoft Why do I pay for WORD when it funct...
2,Negative,"CSGO matchmaking is so full of closet hacking,..."
3,Neutral,Now the President is slapping Americans in the...
4,Negative,Hi @EAHelp I’ve had Madeleine McCann in my cel...


In [ ]:
import re
import string

def preprocess_text(text):

    # Remove URLs https://www.
    ulr_pattern = re.compile(r'https?://\S+|www\.\S+')
    text = ulr_pattern.sub(r'', text)

    # Remove HTML tags
    html_pattern = re.compile('<.*?>')
    text = html_pattern.sub(r'', text)
    # remove puncs and digits
    replace_chars = list(string.punctuation + string.digits)
    for char in replace_chars:
        text = text.replace(char, " ")

    # remove emoji
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U0001F1F2-\U0001F1F4"  # Macau flag
        u"\U0001F1E6-\U0001F1FF"  # flags
        u"\U0001F600-\U0001F64F"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U0001f926-\U0001f937"
        u"\U0001F1F2"
        u"\U0001F1F4"
        u"\U0001F620"
        u"\u200d"
        u"\u2640-\u2642"
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r" ", text)

    # normalize whitespace
    text = " ".join(text.split())

    # lowercasing
    text = text.lower()
    return text

In [ ]:
train_df['text'] = train_df['text'].apply(lambda x: preprocess_text(x))
test_df['text'] = test_df['text'].apply(lambda x: preprocess_text(x))

In [ ]:
train_df.head()

,label,text
0,Positive,i am coming to the borders and i will kill you...
1,Positive,im getting on borderlands and i will kill you all
2,Positive,im coming on borderlands and i will murder you...
3,Positive,im getting on borderlands and i will murder yo...
4,Positive,im getting into borderlands and i can murder y...


In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
train_df['label_code'] = le.fit_transform(train_df['label'])
test_df['label_code'] = le.transform(test_df['label'])
print(le.classes_)


['Irrelevant' 'Negative' 'Neutral' 'Positive']


In [ ]:
test_df.head()

,label,text,label_code
0,Neutral,bbc news amazon boss jeff bezos rejects claims...,2
1,Negative,microsoft why do i pay for word when it functi...,1
2,Negative,csgo matchmaking is so full of closet hacking ...,1
3,Neutral,now the president is slapping americans in the...,2
4,Negative,hi eahelp i’ve had madeleine mccann in my cell...,1


In [ ]:
label_mapping = {int(index): label for index, label in enumerate(le.classes_)}

# Lưu vào file JSON
with open('/content/drive/MyDrive/Colab Notebooks/sentiment_analysis/model/label_mapping.json', 'w') as f:
    json.dump(label_mapping, f)

In [ ]:
# from torchtext.data.utils import get_tokenizer
# from torchtext.vocab import build_vocab_from_iterator
# import json
# import pickle


# tokenizer = get_tokenizer('basic_english')
# max_vocab_size = 30725
# def yield_tokens(sentences, tokenizer):
#     for sentence in sentences:
#         yield tokenizer(sentence)

# vocab = build_vocab_from_iterator(
#     iterator=yield_tokens(sentences=train_df['text'], tokenizer=tokenizer),
#     specials=['<unk>', '<pad>'],
#     max_tokens=max_vocab_size
# )
# vocab.set_default_index(vocab['<unk>'])



In [ ]:
# # Save vocab
# with open('/content/drive/MyDrive/Colab Notebooks/sentiment_analysis/model/vocab.pkl', 'wb') as f:
#     pickle.dump(vocab, f)

# with open('/content/drive/MyDrive/Colab Notebooks/sentiment_analysis/model/vocab.pkl', 'rb') as f:
#   vocab = pickle.load(f)


In [ ]:
from torchtext.data.functional import to_map_style_dataset

def prepare_dataset(df):
    for index, row in df.iterrows():
        sentence = row['text']
        encoded_sentence = vocab(tokenizer(sentence))
        label = row['label']
        yield encoded_sentence, label

train_dataset = prepare_dataset(train_df)
train_dataset = to_map_style_dataset(train_dataset)

test_dataset = prepare_dataset(test_df)
test_dataset = to_map_style_dataset(test_dataset)

In [ ]:
test_dataset[0], test_df['text'][0], test_df['label'][0]

(([1731,
   178,
   81,
   1695,
   1919,
   2873,
   5621,
   1602,
   326,
   5436,
   34,
   6,
   1958,
   8727,
   1731,
   221,
   335,
   178,
   3178,
   22430],
  2),
 'bbc news amazon boss jeff bezos rejects claims company acted like a drug dealer bbc co uk news av busine…',
 2)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, BatchNormalization
from tensorflow.keras.regularizers import l2
import spacy

In [ ]:
# Separate features and labels for training data
train_texts = train_df['text'].values
train_labels = train_df['label'].values

# Separate features and labels for test data
test_texts = test_df['text'].values
test_labels = test_df['label'].values

In [ ]:
# Initialize the label encoder
label_encoder = LabelEncoder()

# Fit and transform the labels
train_labels_encoded = label_encoder.fit_transform(train_labels)
test_labels_encoded = label_encoder.transform(test_labels)

In [ ]:
# Initialize the tokenizer
tokenizer1 = Tokenizer()  # You can adjust the number of words

# Fit the tokenizer on the training texts
tokenizer1.fit_on_texts(train_texts)
tokenizer1.fit_on_texts(test_texts)

In [ ]:
# Convert texts to sequences
train_sequences = tokenizer1.texts_to_sequences(train_texts)
test_sequences = tokenizer1.texts_to_sequences(test_texts)

In [ ]:
# Maximum Length in X_train_sequences
maxlen = max(len(tokens) for tokens in train_sequences)
print("Maximum sequence length (maxlen):", maxlen)

Maximum sequence length (maxlen): 166


In [ ]:
train_padded = pad_sequences(train_sequences, maxlen=maxlen, padding='post')
test_padded = pad_sequences(test_sequences, maxlen=maxlen, padding='post')

In [ ]:
train_padded.shape

(71655, 166)

In [ ]:
# Embedding Input Size
input_size = np.max(train_padded) + 1
input_size

30722

In [ ]:
type(train_padded[0])

numpy.ndarray

In [ ]:
root_path = '/content/drive/MyDrive/Colab Notebooks/sentiment_analysis'
from torchtext.data.utils import get_tokenizer
import json
import pickle
import re
import string
def clean_text(text):

    # Remove URLs https://www.
    ulr_pattern = re.compile(r'https?://\S+|www\.\S+')
    text = ulr_pattern.sub(r'', text)

    # Remove HTML tags
    html_pattern = re.compile('<.*?>')
    text = html_pattern.sub(r'', text)
    # remove puncs and digits
    replace_chars = list(string.punctuation + string.digits)
    for char in replace_chars:
        text = text.replace(char, " ")

    # remove emoji
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U0001F1F2-\U0001F1F4"  # Macau flag
        u"\U0001F1E6-\U0001F1FF"  # flags
        u"\U0001F600-\U0001F64F"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U0001f926-\U0001f937"
        u"\U0001F1F2"
        u"\U0001F1F4"
        u"\U0001F620"
        u"\u200d"
        u"\u2640-\u2642"
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r" ", text)

    # normalize whitespace
    text = " ".join(text.split())

    # lowercasing
    text = text.lower()
    return text
def label_mapping(index):
  with open(os.path.join(root_path, 'model/label_mapping.json'), 'r') as f:
      label_map = json.load(f)
  return label_map[str(index)]
def get_vocab():
  with open(os.path.join(root_path, 'model/vocab.pkl'), 'rb') as f:
      vocab = pickle.load(f)
  return vocab


def text_encode(text, maxlen = 166):
    text = clean_text(text)
    tokenizer = get_tokenizer('basic_english')
    vocab = get_vocab()
    encoded_sentence = vocab(tokenizer(text))
    if len(encoded_sentence) < maxlen:
        encoded_sentence = encoded_sentence + [0] * (maxlen - len(encoded_sentence))
    else:
        encoded_sentence = encoded_sentence[:maxlen]
    return np.array(encoded_sentence)







In [ ]:
tokenizer = get_tokenizer('basic_english')
vocab = get_vocab()


def text_encode1(text, maxlen = 166, ):
    text = clean_text(text)
    encoded_sentence = vocab(tokenizer(text))
    if len(encoded_sentence) < maxlen:
        encoded_sentence = encoded_sentence + [0] * (maxlen - len(encoded_sentence))
    else:
        encoded_sentence = encoded_sentence[:maxlen]
    return np.array(encoded_sentence)


train_encoded_input = [text_encode1(t) for t in train_df['text']]
test_encoded_input =  [text_encode1(t) for t in test_df['text']]

In [ ]:
train_encoded_input = np.array(train_encoded_input)
test_encoded_input = np.array(test_encoded_input)

In [ ]:
train_labels_encoded = np.array(train_df['label_code'])
test_labels_encoded = np.array(test_df['label_code'])

In [ ]:
test_labels_encoded

array([2, 1, 1, 2, 1, 3, 3, 3, 1, 3, 3, 1, 2, 1, 3, 3, 1, 3, 1, 1, 2, 0,
       1, 2, 2, 1, 0, 0, 1, 3, 3, 1, 3, 1, 2, 2, 0, 3, 2, 3, 2, 2, 2, 3,
       2, 1, 1, 1, 2, 3, 1, 1, 3, 3, 3, 3, 3, 1, 0, 1, 3, 3, 0, 1, 2, 1,
       0, 2, 1, 3, 1, 1, 3, 3, 0, 3, 0, 2, 2, 2, 3, 3, 2, 3, 2, 1, 0, 1,
       2, 2, 1, 3, 0, 0, 1, 1, 1, 2, 3, 2, 1, 3, 3, 2, 3, 2, 3, 1, 2, 2,
       2, 1, 2, 1, 2, 2, 3, 3, 2, 1, 1, 3, 1, 2, 1, 3, 2, 1, 2, 0, 3, 2,
       3, 3, 0, 2, 2, 0, 0, 0, 2, 2, 0, 0, 0, 3, 2, 3, 0, 3, 1, 2, 2, 2,
       0, 2, 1, 2, 3, 1, 2, 1, 0, 0, 0, 2, 1, 1, 1, 3, 3, 3, 2, 2, 3, 0,
       2, 2, 2, 3, 2, 1, 1, 2, 3, 3, 0, 0, 2, 3, 3, 2, 0, 2, 1, 1, 1, 1,
       3, 2, 2, 3, 3, 3, 3, 1, 3, 3, 0, 2, 0, 1, 1, 0, 0, 1, 3, 3, 1, 0,
       1, 3, 3, 1, 0, 0, 3, 3, 1, 3, 0, 2, 0, 0, 1, 2, 2, 3, 1, 0, 0, 3,
       3, 0, 0, 2, 3, 1, 1, 3, 3, 3, 3, 2, 2, 3, 1, 2, 3, 2, 1, 2, 2, 1,
       3, 3, 0, 1, 2, 0, 3, 2, 0, 1, 2, 1, 3, 3, 1, 1, 1, 3, 1, 2, 3, 2,
       2, 1, 3, 1, 3, 1, 0, 2, 2, 3, 1, 2, 1, 0, 3,

In [ ]:
train_encoded_input

array([[   3,  107,  405, ...,    0,    0,    0],
       [ 307,  170,   14, ...,    0,    0,    0],
       [ 307,  405,   14, ...,    0,    0,    0],
       ...,
       [  26, 1908,    2, ...,    0,    0,    0],
       [  26, 1908,  735, ...,    0,    0,    0],
       [  26,   34,    2, ...,    0,    0,    0]])

In [ ]:
# Define the model
model1 = Sequential()
# Add an embedding layer
model1.add(Embedding(input_dim=input_size, output_dim=100, input_shape=(56,)))
# Add a bidirectional LSTM layer with 128 units
model1.add(Bidirectional(LSTM(128, kernel_regularizer=l2(0.1), return_sequences=True, recurrent_regularizer=l2(0.1))))
model1.add(BatchNormalization())
model1.add(Dropout(0.5))
# Add another LSTM layer
model1.add(Bidirectional(LSTM(64, kernel_regularizer=l2(0.01), recurrent_regularizer=l2(0.01))))
# Add batch normalization layer
model1.add(BatchNormalization())
model1.add(Dropout(0.5))
# Add a dense layer with 64 units and ReLU activation
model1.add(Dense(64, activation='relu', kernel_regularizer=l2(0.01)))
# Add dropout regularization
model1.add(Dropout(0.5))
# Add the output layer with 5 units for 5 labels and softmax activation
model1.add(Dense(5, activation='softmax'))

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
import torch
model1.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
history = model1.fit(
    train_encoded_input,
    train_labels_encoded ,
    validation_data=(test_encoded_input,test_labels_encoded),
    epochs=20,
)


Epoch 1/20
2240/2240 ━━━━━━━━━━━━━━━━━━━━ 71s 29ms/step - accuracy: 0.4576 - loss: 8.1689 - val_accuracy: 0.2943 - val_loss: 2.3937
Epoch 2/20
2240/2240 ━━━━━━━━━━━━━━━━━━━━ 64s 28ms/step - accuracy: 0.7923 - loss: 0.6920 - val_accuracy: 0.2382 - val_loss: 2.5654
Epoch 3/20
2240/2240 ━━━━━━━━━━━━━━━━━━━━ 63s 28ms/step - accuracy: 0.8384 - loss: 0.5649 - val_accuracy: 0.2392 - val_loss: 2.9205
Epoch 4/20
2240/2240 ━━━━━━━━━━━━━━━━━━━━ 63s 28ms/step - accuracy: 0.8617 - loss: 0.4977 - val_accuracy: 0.2823 - val_loss: 2.8637
Epoch 5/20
2240/2240 ━━━━━━━━━━━━━━━━━━━━ 63s 28ms/step - accuracy: 0.8766 - loss: 0.4494 - val_accuracy: 0.2583 - val_loss: 3.3428
Epoch 6/20
2240/2240 ━━━━━━━━━━━━━━━━━━━━ 63s 28ms/step - accuracy: 0.8876 - loss: 0.4147 - val_accuracy: 0.2743 - val_loss: 3.3335
Epoch 7/20
2240/2240 ━━━━━━━━━━━━━━━━━━━━ 63s 28ms/step - accuracy: 0.8980 - loss: 0.3833 - val_accuracy: 0.2533 - val_loss: 3.9353
Epoch 8/20
2240/2240 ━━━━━━━━━━━━━━━━━━━━ 63s 28ms/step - accuracy: 0.9005 -

In [ ]:

model1.save_weights('/content/drive/MyDrive/Colab Notebooks/sentiment_analysis/modelsv3.weights.h5')
model1.save('/content/drive/MyDrive/Colab Notebooks/sentiment_analysis/model/modelsv3.keras')
model1.save('/content/drive/MyDrive/Colab Notebooks/sentiment_analysis/model/modelsv3.h5')

